# PFEM/Transolver Training -- B1 x {NH, MR, AB}, B2 x {NH, MR, AB}
**Colab GPU runtime required (Runtime > Change runtime type > GPU)**

Runs the full PFEM pipeline (Wang et al., "Pretrain finite element method",
JMPS 214 (2026) 106682) for all 6 benchmark cases: B1 (unit square,
top-edge traction, fixed bottom) and B2 (quarter ring, R_in=1/R_out=2,
internal pressure, symmetry BCs), each with three hyperelastic energy
densities (Neo-Hookean, Mooney-Rivlin, Arruda-Boyce).

For each case this notebook:
1. Generates a FEM ground-truth dataset (Total-Lagrangian Newton-Raphson,
   Q4 elements) via `omar_pfem/data/data_generate_B{1,2}.py`, parallelized
   across CPU cores -- skipped if the dataset already exists on disk.
2. Converts it to the Transolver NPZ format via `convert_B{1,2}_quad.py`.
3. Trains a physics-informed Transolver (`omar_pfem/train_B{1,2}.py`) by
   minimizing total potential energy (Pi = U - W, no labeled-data loss),
   matching PFEM's own reference hyperparameters (n_hidden=256, n_layers=4,
   n_heads=8, slice_num=128, batch_size=1, epochs=10000).

**Scale note**: PFEM's own reference script defaults to ntrain=800,
ntest=200, 10000 epochs, batch_size=1 -- i.e. up to 8,000,000 single-sample
gradient steps per case, 48,000,000 across all 6. This is a genuinely
large campaign, on both ends of the pipeline:
- **Data generation**: measured on a 4-core CPU at the reference 21x21 Q4
  mesh, generating each sample (a 10-step Newton-Raphson solve) took
  ~8s/sample with 4 parallel workers -- so a 1000-sample dataset
  (NTRAIN+NTEST) is roughly 2-2.5 CPU-hours per case, ~14 hours for all 6.
  Colab's own CPU allocation may differ from this benchmark.
- **Training**: 10000 epochs x 800 training samples x batch_size=1 is a
  genuinely large number of individual gradient steps; whether it
  completes in one session depends entirely on Colab's GPU throughput,
  which this notebook doesn't try to predict in advance.

Given that, it will very likely NOT finish within one Colab session.
That's expected and handled: every case checkpoints its model every
`SAVE_EVERY` epochs and **resumes from its own latest checkpoint** if this
notebook is re-run (whether because the runtime disconnected mid-case, or
because you're continuing across multiple sessions) -- so re-running
`Runtime > Run all` after a disconnect always makes forward progress
instead of starting over. Finished cases (a `model_final.pt` on disk) are
skipped entirely, and finished datasets (an NPZ already on disk) are never
regenerated.

If you'd rather trade fidelity for a faster full 6-case pass, lower
`TARGET_SAMPLES` / `EPOCHS` in the config cell below -- everything else
adapts automatically.


## Cell 1 - Install dependencies

Colab's preinstalled `torch` already has CUDA support -- it is deliberately
NOT reinstalled here (a bare `pip install torch` risks silently replacing
it with a CPU-only wheel). Only the packages PFEM's Transolver model and
this pipeline actually need on top of Colab's base image are installed:
`einops`/`timm` (Transolver architecture), `h5py` (FEM dataset storage),
`jax` (autodiff-derived PK1 stress/tangent for Mooney-Rivlin/Arruda-Boyce
in the FEM generator -- CPU-only use, small dense tensors, no GPU needed).

Uses `{sys.executable} -m pip` rather than bare `!pip` -- on some Colab
runtimes the shell's `pip` has been observed to resolve to a different
Python than the notebook kernel itself, which makes packages "install
successfully" yet still fail to import.


In [ ]:
import sys
!{sys.executable} -m pip install -q einops timm h5py jax tqdm
print('Done - continue to Cell 2')


## Cell 2 - Clone the repo

If the repo is private, paste a GitHub personal access token as the value
of `GITHUB_TOKEN` below. Leave it as `""` if the repo is public -- never
type a literal `<TOKEN>` placeholder into the URL (`<`/`>` are bash
redirection operators and will break the clone before git even runs).

Always does a clean re-clone of the CODE (removes any previous
`/content/OMAR` first) so a broken partial clone can't linger -- but
generated datasets and results live elsewhere (local disk / Google Drive,
see Cells 3-4), **outside** the cloned repo, so re-cloning never discards
training progress.


In [ ]:
import os
import shutil
import sys

os.chdir('/content')

GITHUB_TOKEN = ""  # <-- paste your token between the quotes if the repo is private; leave "" if public
BRANCH = "claude/claude-code-question-d307wp"
REPO_URL = (f"https://{GITHUB_TOKEN}@github.com/suhibamro/omar.git" if GITHUB_TOKEN
            else "https://github.com/suhibamro/omar.git")

if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')

!git clone -b {BRANCH} {REPO_URL} /content/OMAR

WORK_DIR = '/content/OMAR/Practical_Examples'
if not os.path.isdir(WORK_DIR):
    raise SystemExit(
        'ERROR: clone failed -- /content/OMAR/Practical_Examples does not exist.\n'
        'Scroll up to the "git clone" output above for the actual error.\n'
        'Common cause: the repo is private and GITHUB_TOKEN is still "".'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

pfem_ok = os.path.isdir(os.path.join(WORK_DIR, 'omar_pfem'))
if pfem_ok:
    print('Clone OK: omar_pfem/ found under Practical_Examples/.')
else:
    raise SystemExit('ERROR: omar_pfem/ not found -- clone looks incomplete or wrong branch.')

import torch
print(f'torch {torch.__version__} | cuda available={torch.cuda.is_available()}', end=' ')
if torch.cuda.is_available():
    print(f'| device={torch.cuda.get_device_name(0)}')
else:
    print()
    print('WARNING: no GPU detected -- go to Runtime > Change runtime type and select a GPU, '
          'then Runtime > Restart session and re-run from Cell 1.')


## Cell 3 - Mount Google Drive (persist training progress across full disconnects)

Colab's local `/content` disk only survives a *reconnect* to the same
runtime -- it does NOT survive a full runtime reset/reassignment (hitting
the session time limit, "Factory reset runtime", or Colab reclaiming an
idle VM). Given this pipeline can realistically run for many hours to
multiple days across several sessions, that's a real risk for the
expensive part of the run: GPU training checkpoints.

So results (`RESULTS_DIR` below) are written to Google Drive, which
survives any of the above. FEM datasets (`DATA_DIR`) stay on local
`/content` disk instead -- deliberately NOT on Drive -- because Drive is
FUSE-mounted (each file write is a network round-trip), and the data
generator does hundreds of small sequential writes into one HDF5 file per
case; on Drive that overhead would meaningfully slow down the part of the
pipeline that's cheap to just regenerate if lost (a bounded few CPU-hours,
deterministic given the same seed), whereas losing GPU training progress
is not cheap to redo. If prompted, click through the Google auth flow --
this notebook will not work with your data without that authorization.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted -- continue to Cell 4')


## Cell 4 - Configuration

Edit these to trade off dataset/training scale against wall-clock time.
`TARGET_SAMPLES` splits into `NTRAIN`/`NTEST` the same way as PFEM's own
reference script (800/200 by default -- lower this if a full 1000-sample
FEM generation pass is too slow on your Colab CPU allocation).


In [ ]:
import os

DATA_DIR = '/content/pfem_data'                                  # local disk: fast, regenerable if lost
RESULTS_DIR = '/content/drive/MyDrive/pfem_run/results'          # Google Drive: durable across full resets
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Dataset scale (PFEM's own reference: ntrain=800, ntest=200) ----
NTRAIN = 800
NTEST = 200
TARGET_SAMPLES = NTRAIN + NTEST   # total FEM samples generated per case

# ---- Mesh resolution (PFEM's own reference default for the beam case) ----
MESH_N = 21   # Nx=Ny=21 for B1; Ntheta=Nr=21 for B2

# ---- Training scale (PFEM's own reference default) ----
EPOCHS = 10000
SAVE_EVERY = 250      # checkpoint + eval cadence (40 checkpoints over a full run)
PRINT_EVERY = 999999  # larger than NTRAIN -> exactly one progress line per epoch

N_WORKERS = os.cpu_count()

GEOMETRIES = ["B1", "B2"]
MATERIALS = ["neo_hookean", "mooney_rivlin", "arruda_boyce"]
CASES = [(g, m) for g in GEOMETRIES for m in MATERIALS]

print(f'CPU workers for data generation: {N_WORKERS}')
print(f'Target samples/case: {TARGET_SAMPLES} (ntrain={NTRAIN}, ntest={NTEST})')
print(f'Epochs/case: {EPOCHS}, save_every={SAVE_EVERY}')
print(f'Cases: {CASES}')


## Cell 5 - Helpers: streaming subprocess runner + per-case pipeline

`run_streaming` runs a command as a subprocess and prints its stdout live
(unbuffered, `python -u`) instead of buffering until the process exits --
essential for a run that may take hours, so progress is visible the whole
time instead of appearing in one dump at the end (or not at all, if the
cell is interrupted).


In [ ]:
import subprocess
import time
import json as _json

def run_streaming(cmd, cwd=None):
    print(f"$ {' '.join(cmd)}")
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(cmd)}")


def case_name(geometry, material):
    return f"{geometry}_{material}"


def run_one_case(geometry, material):
    name = case_name(geometry, material)
    h5_dir = os.path.join(DATA_DIR, f"fem_{name}")
    npz_dir = os.path.join(DATA_DIR, f"npz_{name}")
    npz_path = os.path.join(npz_dir, "hyperelastic_training_data_q4.npz")
    out_dir = os.path.join(RESULTS_DIR, name)
    final_model = os.path.join(out_dir, "model_final.pt")

    print(f"\n{'='*80}\n===== CASE: {name} =====\n{'='*80}")

    if os.path.exists(final_model):
        print(f"[{name}] model_final.pt already exists -- case fully done, skipping entirely.")
        return

    t0 = time.time()

    # ---- Step 1: FEM dataset generation (skipped if already present) ----
    if os.path.exists(npz_path):
        print(f"[{name}] NPZ dataset already exists at {npz_path}, skipping generation.")
    else:
        if geometry == "B1":
            gen_module = "omar_pfem.data.data_generate_B1"
            gen_args = ["--Nx", str(MESH_N), "--Ny", str(MESH_N)]
            conv_module = "omar_pfem.data.convert_B1_quad"
        else:
            gen_module = "omar_pfem.data.data_generate_B2"
            gen_args = ["--Ntheta", str(MESH_N), "--Nr", str(MESH_N)]
            conv_module = "omar_pfem.data.convert_B2_quad"

        run_streaming([
            sys.executable, "-u", "-m", gen_module,
            "--num_index", "1", "--num_samples", str(TARGET_SAMPLES),
            *gen_args, "--material", material,
            "--n_workers", str(N_WORKERS),
            "--out_dir", h5_dir,
        ], cwd=WORK_DIR)

        run_streaming([
            sys.executable, "-u", "-m", conv_module,
            "--h5_dir", h5_dir, "--out_dir", npz_dir,
        ], cwd=WORK_DIR)

        print(f"[{name}] Dataset ready after {time.time()-t0:.0f}s")

    # ---- Step 2: Training (resumes from its own latest checkpoint) ----
    train_module = "omar_pfem.train_B1" if geometry == "B1" else "omar_pfem.train_B2"
    t1 = time.time()
    run_streaming([
        sys.executable, "-u", "-m", train_module,
        "--path", npz_path,
        "--material", material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--epochs", str(EPOCHS),
        "--save_every", str(SAVE_EVERY),
        "--print_every", str(PRINT_EVERY),
        "--out_dir", out_dir,
    ], cwd=WORK_DIR)
    print(f"[{name}] Training finished after {time.time()-t1:.0f}s (total {time.time()-t0:.0f}s)")


## Cell 6 - Run all 6 cases

Cases run one at a time (each already uses all CPU cores for its own data
generation and the whole GPU for its own training, so there's no benefit
to overlapping them). If this cell is interrupted (disconnect, manual
stop) and re-run, already-finished cases are skipped instantly and the
case that was interrupted resumes from its latest epoch checkpoint instead
of restarting -- see Cell 3's config and the "Scale note" in the intro.


In [ ]:
for geometry, material in CASES:
    run_one_case(geometry, material)

print("\nAll cases either completed or already were -- see per-case status above.")


## Cell 7 - Summary: metrics + loss curves across all 6 cases

Reads each case's `metrics_history.json` (written every `SAVE_EVERY`
epochs during training) -- works even for cases that are still mid-run or
were only partially completed before a disconnect.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
summary_rows = []

for ax, (geometry, material) in zip(axes.flat, CASES):
    name = case_name(geometry, material)
    metrics_path = os.path.join(RESULTS_DIR, name, "metrics_history.json")
    if not os.path.exists(metrics_path):
        ax.set_title(f"{name}\n(no data yet)")
        continue

    with open(metrics_path) as f:
        history = _json.load(f)
    if not history:
        ax.set_title(f"{name}\n(empty history)")
        continue

    epochs = [h["epoch"] for h in history]
    l2u = [h["mean_rel_L2_u"] for h in history]
    l2v = [h["mean_rel_L2_v"] for h in history]

    ax.semilogy(epochs, l2u, label="Rel L2(u)")
    ax.semilogy(epochs, l2v, label="Rel L2(v)")
    ax.set_title(f"{name} (epoch {epochs[-1]}/{EPOCHS})")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

    summary_rows.append((name, epochs[-1], l2u[-1], l2v[-1]))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "all_cases_loss_curves.png"), dpi=150)
plt.show()

print(f"{'case':<20s} {'last epoch':>10s} {'RelL2(u)':>12s} {'RelL2(v)':>12s}")
for name, ep, u, v in summary_rows:
    print(f"{name:<20s} {ep:>10d} {u:>12.3e} {v:>12.3e}")


## Cell 8 - Zip and download all results (optional)

Results already live on Google Drive (`RESULTS_DIR`), so this is just a
convenience if you want a local zip copy too -- not required for safety.


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/pfem_results', 'zip', RESULTS_DIR)
print(f"Archived {RESULTS_DIR} -> {archive_path}")
files.download(archive_path)
